In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("stg-yellow-taxi-data-eda")
    .config(
        "spark.jars.packages",
        ",".join([
            "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.9.0",
            "org.apache.iceberg:iceberg-aws-bundle:1.9.0",
            "org.apache.hadoop:hadoop-aws:3.3.4",
            "com.amazonaws:aws-java-sdk-bundle:1.12.262",
        ]),
    )
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
    .config("spark.sql.defaultCatalog", "lakehouse")
    .config("spark.sql.catalog.lakehouse", "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.lakehouse.type", "rest")
    .config("spark.sql.catalog.lakehouse.uri", "http://localhost:8181")
    .config("spark.sql.catalog.lakehouse.warehouse", "s3://warehouse/")
    .config("spark.sql.catalog.lakehouse.io-impl", "org.apache.iceberg.aws.s3.S3FileIO")
    .config("spark.sql.catalog.lakehouse.client.region", "us-east-1")
    .config("spark.sql.catalog.lakehouse.s3.endpoint", "http://localhost:9000")
    .config("spark.sql.catalog.lakehouse.s3.path-style-access", "true")
    .config("spark.sql.catalog.lakehouse.s3.access-key-id", "admin")
    .config("spark.sql.catalog.lakehouse.s3.secret-access-key", "binhdeptrai")
    .config("spark.hadoop.fs.s3a.endpoint", "http://localhost:9000")
    .config("spark.hadoop.fs.s3a.access.key", "admin")
    .config("spark.hadoop.fs.s3a.secret.key", "binhdeptrai")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .getOrCreate()
)

In [ ]:
spark.sql("SHOW NAMESPACES IN lakehouse").show()

In [ ]:
spark.table("lakehouse.silver.yellow_trips").limit(5).show()

### Validating Silver Data

In [5]:
# Row counts by year
spark.sql("""
    SELECT
        YEAR(pickup_ts) AS year,
        COUNT(*) AS trip_count
    FROM lakehouse.silver.yellow_trips
    GROUP BY YEAR(pickup_ts)
    ORDER BY year
""").show()

+----+----------+
|year|trip_count|
+----+----------+
|2003|         6|
|2008|         8|
|2009|       129|
|2011|  13393301|
|2012|  13074637|
|2013|  15748793|
|2014|  14618614|
|2015|  13154312|
|2016|  11131166|
|2017|   8587811|
|2018|   7784304|
|2019|   6413046|
|2020|   1542560|
|2021|   3263973|
|2022|   3180721|
|2023|     32129|
+----+----------+



In [8]:
# Row count by month
spark.sql("""
    SELECT
        YEAR(pickup_ts) AS year,
        MONTH(pickup_ts) AS month,
        COUNT(*) AS trip_count
    FROM lakehouse.silver.yellow_trips
    GROUP BY YEAR(pickup_ts), MONTH(pickup_ts)
    ORDER BY year, month
""").show(200)

+----+-----+----------+
|year|month|trip_count|
+----+-----+----------+
|2003|    1|         6|
|2008|   12|         8|
|2009|    1|       129|
|2011|    1|  13223971|
|2011|    2|    169330|
|2012|    2|  12891757|
|2012|    3|    182880|
|2013|    3|  15599468|
|2013|    4|    149325|
|2014|    4|  14424473|
|2014|    5|    194141|
|2015|    5|  13042288|
|2015|    6|    112024|
|2016|    6|  10986369|
|2016|    7|    144797|
|2017|    7|   8483545|
|2017|    8|    104266|
|2018|    1|         1|
|2018|    7|         9|
|2018|    8|   7695542|
|2018|    9|     88703|
|2018|   10|        28|
|2018|   11|        12|
|2018|   12|         9|
|2019|    1|        10|
|2019|    2|         1|
|2019|    3|         2|
|2019|    4|         4|
|2019|    5|         2|
|2019|    8|         6|
|2019|    9|   6345421|
|2019|   10|     67520|
|2019|   11|        50|
|2019|   12|        30|
|2020|    1|        19|
|2020|    2|        19|
|2020|    3|         3|
|2020|    4|         3|
|2020|    5|    